In [1]:
import cv2
import albumentations as A
import os
import random
import shutil

In [3]:
import albumentations as A
import cv2
import os

# Percorsi
raw_images_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\raw\images'
raw_labels_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\raw\labels'
augmented_images_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\augmented\images'
augmented_labels_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\augmented\labels'

# Crea directory augmented se non esistono
os.makedirs(augmented_images_dir, exist_ok=True)
os.makedirs(augmented_labels_dir, exist_ok=True)

# Definisci la pipeline di augmentazione con tecniche aggiuntive
augmentation_pipeline = A.Compose([
    A.RandomBrightnessContrast(p=0.5),  # Aumento del contrasto
    A.Sharpen(p=0.5),  # Aumenta la nitidezza
    A.GaussianBlur(blur_limit=3, p=0.3),  # Aggiunge sfocatura leggera
    A.Resize(640, 640),  # Ridimensiona l'immagine per adattarla alla risoluzione standard
    A.HorizontalFlip(p=0.5),  # Flip orizzontale
HueSaturationValue(p=0.3),  # Adjust hue and saturation
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),  # Rotazione, scala e spostamento
    A.RandomCrop(height=480, width=480, p=0.3),  # Ritaglio casuale
    A.GaussNoise(p=0.2)  # Aggiunge rumore per aumentare la robustezza
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# Funzione per caricare i bounding boxes da file .txt
def load_bboxes(label_file):
    with open(label_file, 'r') as f:
        bboxes = []
        class_labels = []
        for line in f.readlines():
            label = line.strip().split()
            class_labels.append(int(label[0]))  # Class ID (0 per olive)
            bbox = list(map(float, label[1:]))  # x_center, y_center, width, height
            bboxes.append(bbox)
    return bboxes, class_labels

# Funzione per salvare i bounding boxes aggiornati
def save_bboxes(label_file, bboxes, class_labels):
    with open(label_file, 'w') as f:
        for bbox, class_label in zip(bboxes, class_labels):
            bbox_str = ' '.join([str(x) for x in bbox])
            f.write(f"{class_label} {bbox_str}\n")

# Applica data augmentation
image_files = [f for f in os.listdir(raw_images_dir) if f.endswith('.jpg')]

for image_file in image_files:
    base_name = os.path.splitext(image_file)[0]
    image_path = os.path.join(raw_images_dir, image_file)
    label_path = os.path.join(raw_labels_dir, base_name + '.txt')

    # Carica immagine e bounding boxes
    image = cv2.imread(image_path)
    bboxes, class_labels = load_bboxes(label_path)

    for i in range(5):  # Genera 3 immagini aumentate per ogni immagine originale
        augmented = augmentation_pipeline(image=image, bboxes=bboxes, class_labels=class_labels)
        augmented_image = augmented['image']
        augmented_bboxes = augmented['bboxes']
        augmented_class_labels = augmented['class_labels']

        # Salva immagine aumentata
        augmented_image_path = os.path.join(augmented_images_dir, f'aug_{i}_{image_file}')
        cv2.imwrite(augmented_image_path, augmented_image)

        # Salva etichetta aumentata
        augmented_label_path = os.path.join(augmented_labels_dir, f'aug_{i}_{base_name}.txt')
        save_bboxes(augmented_label_path, augmented_bboxes, augmented_class_labels)

print("Data augmentation completata con successo!")


Data augmentation completata con successo!


In [ ]:
# Percorsi della directory train
train_images_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\train\images'
train_labels_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\train\labels'

# Crea le directory se non esistono
os.makedirs(train_images_dir, exist_ok=True)
os.makedirs(train_labels_dir, exist_ok=True)

# Copia i file raw nella directory train
for file_name in os.listdir(raw_images_dir):
    shutil.copy(os.path.join(raw_images_dir, file_name), train_images_dir)

for file_name in os.listdir(raw_labels_dir):
    shutil.copy(os.path.join(raw_labels_dir, file_name), train_labels_dir)

# Copia i file aumentati nella directory train
for file_name in os.listdir(augmented_images_dir):
    shutil.copy(os.path.join(augmented_images_dir, file_name), train_images_dir)

for file_name in os.listdir(augmented_labels_dir):
    shutil.copy(os.path.join(augmented_labels_dir, file_name), train_labels_dir)


In [6]:
import os
import random
import shutil

# Percorsi della directory di origine (immagini e annotazioni nella cartella "total")
source_images_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\total\images'
source_labels_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\total\labels'

# Creazione delle directory per train, val, e test (nella cartella train_split)
train_images_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\train_split\train\images'
train_labels_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\train_split\train\labels'
val_images_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\train_split\val\images'
val_labels_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\train_split\val\labels'
test_images_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\train_split\test\images'
test_labels_dir = r'C:\Users\Bartolo\OLIVEDETECTIONPROJECT\neural_network\data\train_split\test\labels'

# Crea le directory se non esistono
os.makedirs(train_images_dir, exist_ok=True)
os.makedirs(train_labels_dir, exist_ok=True)
os.makedirs(val_images_dir, exist_ok=True)
os.makedirs(val_labels_dir, exist_ok=True)
os.makedirs(test_images_dir, exist_ok=True)
os.makedirs(test_labels_dir, exist_ok=True)

# Ottieni la lista delle immagini
image_files = [f for f in os.listdir(source_images_dir) if f.endswith('.jpg')]

# Mescola le immagini
random.shuffle(image_files)

# Suddivisione del dataset
train_count = int(len(image_files) * 0.8)  # 70% per train
val_count = int(len(image_files) * 0.1)    # 20% per val

train_files = image_files[:train_count]
val_files = image_files[train_count:train_count + val_count]
test_files = image_files[train_count + val_count:]  # 10% per test

# Funzione per spostare immagini e annotazioni nelle nuove cartelle
def move_files(file_list, split_type, image_dir, label_dir):
    for file_name in file_list:
        base_name = os.path.splitext(file_name)[0]

        # Sposta l'immagine
        shutil.move(os.path.join(source_images_dir, file_name), os.path.join(image_dir, file_name))

        # Sposta l'annotazione
        label_file = base_name + '.txt'
        shutil.move(os.path.join(source_labels_dir, label_file), os.path.join(label_dir, label_file))

# Sposta i file nelle rispettive cartelle (train, val, test)
move_files(train_files, 'train', train_images_dir, train_labels_dir)
move_files(val_files, 'val', val_images_dir, val_labels_dir)
move_files(test_files, 'test', test_images_dir, test_labels_dir)

print("Suddivisione dei file completata con successo!")


Suddivisione dei file completata con successo!


In [4]:
import os
import shutil

# Percorsi alle directory
raw_images_dir = r'C:\Users\Bartolo\OlivesDetectionProject\neural_network\data\raw\images'
raw_labels_dir = r'C:\Users\Bartolo\OlivesDetectionProject\neural_network\data\raw\labels'
augmented_images_dir = r'C:\Users\Bartolo\OlivesDetectionProject\neural_network\data\augmented\images'
augmented_labels_dir = r'C:\Users\Bartolo\OlivesDetectionProject\neural_network\data\augmented\labels'
total_images_dir = r'C:\Users\Bartolo\OlivesDetectionProject\neural_network\data\total\images'
total_labels_dir = r'C:\Users\Bartolo\OlivesDetectionProject\neural_network\data\total\labels'

# Crea la directory totale se non esiste
os.makedirs(total_images_dir, exist_ok=True)
os.makedirs(total_labels_dir, exist_ok=True)

# Funzione per copiare file da una directory a un'altra
def copy_files(src_dir, dst_dir):
    for file_name in os.listdir(src_dir):
        src_file = os.path.join(src_dir, file_name)
        dst_file = os.path.join(dst_dir, file_name)
        if os.path.isfile(src_file):
            shutil.copy(src_file, dst_file)

# Copia i file dalle cartelle 'images' e 'labels' di raw e augmented
copy_files(raw_images_dir, total_images_dir)
copy_files(augmented_images_dir, total_images_dir)
copy_files(raw_labels_dir, total_labels_dir)
copy_files(augmented_labels_dir, total_labels_dir)

print("Directory 'total' creata e file copiati con successo.")


Directory 'total' creata e file copiati con successo.


In [1]:
import os
import json
from PIL import Image

# Percorsi alle cartelle delle immagini e delle etichette YOLO
images_dir = r'C:\Users\Bartolo\OlivesDetectionProject\data\total\images'
labels_dir = r'C:\Users\Bartolo\OlivesDetectionProject\data\total\labels'

# Crea un dizionario vuoto per il formato COCO
coco_output = {
    "info": {
        "year": 2024,
        "version": "1.0",
        "description": "Dataset di olive convertito da YOLO a COCO",
        "contributor": "Nome",
        "date_created": "2024-09-09"
    },
    "licenses": [
        {
            "id": 1,
            "name": "Nome Licenza",
            "url": "URL della licenza"
        }
    ],
    "images": [],
    "annotations": [],
    "categories": [
        {"id": 1, "name": "verde", "supercategory": "olive"},
        {"id": 2, "name": "nero", "supercategory": "olive"},
        {"id": 3, "name": "misto", "supercategory": "olive"}
    ]
}

# Funzione per convertire le coordinate YOLO in COCO (x_centro, y_centro, larghezza, altezza -> xmin, ymin, width, height)
def yolo_to_coco_bbox(x_center, y_center, width, height, img_width, img_height):
    x_min = (x_center - width / 2) * img_width
    y_min = (y_center - height / 2) * img_height
    bbox_width = width * img_width
    bbox_height = height * img_height
    return [x_min, y_min, bbox_width, bbox_height]

# Cicla attraverso tutte le immagini
image_id = 0
annotation_id = 0

for image_filename in os.listdir(images_dir):
    if image_filename.endswith(".jpg") or image_filename.endswith(".png"):
        image_path = os.path.join(images_dir, image_filename)
        label_path = os.path.join(labels_dir, image_filename.replace(".jpg", ".txt").replace(".png", ".txt"))

        # Apri l'immagine per ottenere le sue dimensioni
        with Image.open(image_path) as img:
            img_width, img_height = img.size

        # Aggiungi l'immagine al dizionario COCO
        image_id += 1
        coco_output["images"].append({
            "id": image_id,
            "file_name": image_filename,
            "width": img_width,
            "height": img_height
        })

        # Apri il file di annotazioni YOLO corrispondente
        if os.path.exists(label_path):
            with open(label_path, "r") as label_file:
                for line in label_file:
                    # Each YOLO annotation line is formatted as: class_id x_center y_center width height
                    parts = line.strip().split()
                    class_id = int(parts[0]) + 1  # In COCO, le classi partono da 1, quindi aggiungiamo 1
                    x_center, y_center, width, height = map(float, parts[1:])

                    # Converti le coordinate YOLO in formato COCO
                    bbox = yolo_to_coco_bbox(x_center, y_center, width, height, img_width, img_height)

                    # Aggiungi l'annotazione al dizionario COCO
                    annotation_id += 1
                    coco_output["annotations"].append({
                        "id": annotation_id,
                        "image_id": image_id,
                        "category_id": class_id,
                        "bbox": bbox,
                        "area": bbox[2] * bbox[3],  # larghezza * altezza
                        "iscrowd": 0
                    })

# Salva il file COCO in formato JSON
output_json_path = r"C:\Users\Bartolo\OlivesDetectionProject\data\train_split\coco_annotations.json"
with open(output_json_path, "w") as output_json_file:
    json.dump(coco_output, output_json_file, indent=4)

print(f"Conversione completata! File COCO salvato in {output_json_path}")


Conversione completata! File COCO salvato in C:\Users\Bartolo\OlivesDetectionProject\data\train_split\coco_annotations.json
